# 🎭 Tamil Speech Emotion Recognition (SER) Pipeline
### தமிழ் பேச்சு உணர்ச்சி அறிதல் மாதிரி (Google Colab GPU Optimized)

This notebook trains a **3-Channel Residual-CNN + Bidirectional LSTM + Attention** deep learning model to accurately detect emotion in spoken Tamil speech across **Anger, Happiness, Sadness, Neutral, Fear, and Surprise**.

#### 🌟 Target Emotions & Authentic Tamil Words:
- 😡 **Angry (கோபம்)**: *"போதும் நிறுத்து! இதை என்னால பொறுத்துக்கவே முடியாது!"*
- 😃 **Happy (மகிழ்ச்சி)**: *"வாவ் சூப்பர், நாம் வெற்றி பெற்று விட்டோம்!"*
- 😢 **Sad (சோகம்)**: *"மனசுக்கு ரொம்ப கஷ்டமா இருக்கு, என்ன சொல்றதுன்னே தெரியல..."*
- 😐 **Neutral (இயல்பு)**: *"வணக்கம், இன்றைய செய்தி அறிக்கையை இப்போது பார்க்கலாம்."*
- 😨 **Fear (பயம்)**: *"அங்க ஏதோ விசித்திரமான சத்தம் கேட்குது... பயமா இருக்கு!"*
- 😲 **Surprised (ஆச்சரியம்)**: *"அப்படியா! நிஜமாவா சொல்றீங்க?! உண்மையிலேயே ஆச்சரியம்!"*

## 1. 🚀 Setup Codebase & Dependencies
*(Automatically clones repository if in Colab and prepares the environment)*

In [ ]:
import os
import sys

# Auto-clone repository if running directly in fresh Colab session
if not os.path.exists("src"):
    print("⬇️ Cloning ML-MODEL repository from GitHub...")
    !git clone https://github.com/kevinjosh10/ML-MODEL.git
    %cd ML-MODEL

# Ensure project root is in Python sys.path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working directory: {os.getcwd()}")

# Install dependencies
print("📦 Installing audio and ML dependencies...")
!pip install -q torchaudio librosa soundfile seaborn matplotlib scikit-learn tqdm fastapi uvicorn python-multipart jinja2 pyngrok gtts

import torch
import torchaudio
print(f"\n🔥 PyTorch Version: {torch.__version__}")
print(f"🎵 Torchaudio Version: {torchaudio.__version__}")
print(f"⚡ CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU: {torch.cuda.get_device_name(0)}")

## 2. ⚙️ Prepare Emotional Tamil Speech Dataset
*(Prepares stratified dataset with authentic phonetic formant acoustics across Anger, Sadness, Happiness, etc.)*

In [ ]:
from src.config import Config
from src.data.dataset import get_data_loaders
from src.data.audio_preprocessing import AudioPreprocessor
from src.models import build_model
from src.training.trainer import Trainer
from src.training.metrics import evaluate_model
from src.utils.visualizer import plot_waveform_and_spectrogram, plot_emotion_probabilities
import IPython.display as ipd
import random
import numpy as np

# Initialize configuration with 3-channel feature extraction
config = Config(
    sample_rate=16000,
    duration=3.0,
    epochs=30,
    batch_size=16,
    learning_rate=1e-3,
    seed=42
)

print("📋 Emotional Tamil Target Categories & Spoken Sentences:")
for k, v in config.emotion_map.items():
    print(f"  • {k.upper():10s}: {v}")

train_loader, val_loader, test_loader, classes = get_data_loaders(config)
print(f"\n📊 Stratified Data Split: {len(train_loader.dataset)} Train | {len(val_loader.dataset)} Val | {len(test_loader.dataset)} Test")

## 3. 🎧 Audio Player & Spectrogram Visualizer
Listen to emotional speech clips (e.g. Angry vs. Happy vs. Sad) and view their frequency spectrograms.

In [ ]:
preprocessor = AudioPreprocessor(config, is_train=False)
sample_emotion = random.choice(config.classes)
sample_files = list((config.data_dir / sample_emotion).glob("*.wav"))

if sample_files:
    sample_path = sample_files[0]
    print(f"🔊 Sample Clip: {sample_path.name}")
    print(f"🏷️ Emotion: {config.plot_labels[sample_emotion]} ({config.emotion_map[sample_emotion]})")
    
    # Play audio inside notebook
    ipd.display(ipd.Audio(str(sample_path)))
    
    # Visualize waveform and Log-Mel Spectrogram
    wf = preprocessor.load_audio(sample_path).squeeze().cpu().numpy()
    features = preprocessor.extract_mel_spectrogram(torch.from_numpy(wf), augment=False)
    mel = features[0].squeeze().cpu().numpy()
    plot_waveform_and_spectrogram(wf, config.sample_rate, mel, title=f"Acoustic Spectrum: {config.plot_labels[sample_emotion]}")

## 4. 🧠 Train the 3-Channel Residual SER Model
Trains the Residual CNN-BiLSTM-Attention network with GPU acceleration, AMP, and Cosine Annealing.

In [ ]:
model = build_model(config)
print("Model Architecture:")
print(model)

trainer = Trainer(model, config, train_loader, val_loader)
trainer.fit()

## 5. 📊 Model Evaluation & Confusion Matrix

In [ ]:
best_checkpoint = config.checkpoint_dir / "best_tamil_ser_model.pth"
if best_checkpoint.exists():
    ckpt = torch.load(best_checkpoint, map_location=config.device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Loaded best checkpoint (Epoch {ckpt.get('epoch', 0)+1}, Best Val Acc: {ckpt.get('accuracy', 0):.2f}%)")
else:
    print("ℹ️ Evaluating current in-memory model weights...")

results = evaluate_model(model, test_loader, config)
print(f"\n🎯 Overall Test Accuracy: {results['accuracy']:.2f}%")

print("\n📋 Classification Report:")
for cls_name, metrics in results['classification_report'].items():
    if isinstance(metrics, dict):
        p = metrics.get('precision', 0) * 100
        r = metrics.get('recall', 0) * 100
        f1 = metrics.get('f1-score', 0) * 100
        print(f"  • {cls_name:25s} | Precision: {p:5.1f}% | Recall: {r:5.1f}% | F1: {f1:5.1f}%")

## 6. 🎙️ Live Tamil Voice Recording & Emotion Prediction
Speak an emotional sentence in Tamil (e.g. *"போதும் நிறுத்து!"* for Anger or *"வாவ் சூப்பர்!"* for Happy) to test predictions in real time!

In [ ]:
from src.utils.audio_recorder import record_audio_in_colab
from src.inference import TamilSERPredictor

# Initialize predictor with best model weights
best_checkpoint = config.checkpoint_dir / "best_tamil_ser_model.pth"
if best_checkpoint.exists():
    predictor = TamilSERPredictor(str(best_checkpoint), config=config)
else:
    predictor = TamilSERPredictor(model, config=config)

# Attempt in-browser microphone recording
print("🎙️ Speak into your microphone now (allow browser mic prompt if asked):")
recorded_file = record_audio_in_colab(filename="my_tamil_voice.wav", duration=3)

# Fallback to test dataset sample if mic is skipped
if not recorded_file or not os.path.exists(recorded_file):
    print("\nℹ️ Microphone was skipped. Testing on an audio sample from the test dataset instead...")
    sample_emotion = random.choice(config.classes)
    sample_candidates = list((config.data_dir / sample_emotion).glob("*.wav"))
    if sample_candidates:
        recorded_file = str(sample_candidates[0])

if recorded_file and os.path.exists(recorded_file):
    print(f"\n🔊 Testing Audio Clip: {recorded_file}")
    ipd.display(ipd.Audio(recorded_file))
    
    prediction = predictor.predict(recorded_file, visualize=True)
    
    print("=" * 55)
    print(f"🎉 Predicted Emotion: {prediction['plot_label']} ({prediction['tamil_label']})")
    print(f"✨ Confidence: {prediction['confidence_percentage']}")
    print("=" * 55)

## 7. 🌐 Launch Live Public Website from Colab
Run this cell to start the **Tamil Speech Emotion AI Web Studio** and generate a live public web link!

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Start FastAPI backend in background
print("🚀 Starting Tamil Speech Emotion AI Web Server...")
proc = subprocess.Popen(["python", "app.py"])
time.sleep(3)

# Expose port 8000 via ngrok public tunnel
try:
    ngrok.kill()
    public_url = ngrok.connect(8000)
    print("=" * 65)
    print("🎉 YOUR LIVE WEBSITE IS ONLINE!")
    print(f"👉 Public Web Link: {public_url}")
    print("=" * 65)
except Exception as e:
    print("Ngrok Note:", e)
    print("\nAlternatively, using localtunnel:")
    !npx localtunnel --port 8000